<a href="https://colab.research.google.com/github/dmitresc/Lua-Repository/blob/main/Aquaria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U sentence-transformers transformers accelerate bitsandbytes pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 63.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompat

In [2]:
from google.colab import files
uploaded = files.upload()

Saving freshwater_aquarium_fish_species.csv to freshwater_aquarium_fish_species.csv


In [3]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cuda


In [5]:
embedder = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=DEVICE
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
csv_path = "freshwater_aquarium_fish_species.csv"

df = pd.read_csv(csv_path, encoding="latin1").fillna("Not specified")

SEARCH_COLS = [
    "name",
    "taxonomy",
    "details",
    "temprange",
    "phRange",
    "tank size",
    "fish size (cm/inches)",
    "fish compatibility"
]

def create_labeled_context(row):
    return f"""
Fish Species: {row['name']}
Scientific Name: {row['taxonomy']}

Details:
{row['details']}

Tank Requirements:
- Temperature: {row['temprange']}
- pH Range: {row['phRange']}
- Minimum Tank Size: {row['tank size']}

Fish Size:
{row['fish size (cm/inches)']}

Compatibility:
{row['fish compatibility']}
"""

df["combined_info"] = df.apply(create_labeled_context, axis=1)

In [7]:
print("Creating embeddings...")

fish_embeddings = embedder.encode(
    df["combined_info"].tolist(),
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

Creating embeddings...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [8]:
LLM_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
def retrieve_context(query, top_k=4):

    query_embedding = embedder.encode(
        f"Aquarium question about freshwater fish care, tank setup, compatibility, temperature, or pH: {query}",
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    hits = util.semantic_search(query_embedding, fish_embeddings, top_k=top_k)[0]

    contexts = []
    for hit in hits:
        if hit["score"] > 0.45:
            contexts.append(df.iloc[hit["corpus_id"]]["combined_info"])

    return "\n\n".join(contexts) if contexts else "No relevant data."

In [21]:
from transformers import TextIteratorStreamer
from threading import Thread

def generate_answer(context, question, chat_history, streamer):
    # Keep the prompt short: Only take the last 3 exchanges
    recent_history = chat_history[-3:]

    history_str = ""
    for message in recent_history:
        role = "USER" if message["role"] == "user" else "AQUARIA"
        history_str += f"### {role}: {message['content']}\n"

    # Refined prompt for Phi-2
    prompt = f"""Instruct: You are AQUARIA, a professional freshwater aquarium expert.
Use the provided FISH DATA to answer the QUESTION.
If the data doesn't contain the answer, provide a safe, general recommendation.

FISH DATA:
{context}

{history_str}
QUESTION: {question}
Output: RESPONSE:"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Configure generation parameters
    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=350,
        temperature=0.6,
        do_sample=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Start generation in a separate thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

In [24]:
import warnings
from transformers import TextIteratorStreamer # Import TextIteratorStreamer

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

print("\n🌊 AQUARIA is ready to serve!")
print("Ask about fish, tank size, pH, compatibility, etc.")
print("Type 'quit' to exit.\n")

chat_history = [] # Initialize chat history

while True:
    user = input("You: ")

    if user.lower() in ["quit", "exit"]:
        break

    # Add user message to history before generating response
    chat_history.append({"role": "user", "content": user})

    context = retrieve_context(user)

    # Initialize streamer for this interaction
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    # Call generate_answer to start the generation in a separate thread
    generate_answer(context, user, chat_history, streamer)

    print("\n🐟", end="") # Start printing the assistant's response
    reply_tokens = []
    for new_token in streamer:
        print(new_token, end="", flush=True) # Print tokens as they are generated
        reply_tokens.append(new_token)
    reply = "".join(reply_tokens)
    print("\n") # Add a newline after the full reply

    # Add assistant message to history after generating response
    chat_history.append({"role": "assistant", "content": reply})


🌊 AQUARIA is ready to serve!
Ask about fish, tank size, pH, compatibility, etc.
Type 'quit' to exit.

You: how big can a ghost knife fish gets? and what is the optimal tank size? just two sentences please.

🐟 The ghost knife fish typically does not exceed 18 inches in length, while suitable tank sizes vary depending on factors such as tank volume per inch of fish, recommended stocking densities by aquarists, and individual fish behavior patterns. Generally speaking, the recommended tank size ranges from 55 to 70 gallons based on these considerations. However, specific recommendations may differ based on other variables, including filtration capacity and equipment available. Always consult reliable resources, guidelines, or seek advice from experienced professionals before setting up your aquatic ecosystem.


You: quit


In [12]:
!pip install -q streamlit
!npm install localtunnel


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 46.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [13]:
!wget -qO- ipv4.icanhazip.com

136.110.55.29


In [29]:
! streamlit run app.py & npx localtunnel --port 8501

⠙

your url is: https://modern-rules-tap.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.110.55.29:8501

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 453/453 [00:09<00:00, 49.01it/s]
Loading weights: 100% 103/103 [00:00<00:00, 899.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Stopping...
^C
